In [3]:
# Cell 1: Imports and environment setup
from pathlib import Path
import re
import sys

import gymnasium as gym
import imageio.v2 as imageio
import torch
from IPython.display import Video
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from stable_baselines3.common.monitor import Monitor

# Resolve project root by searching upward for the folder that contains `vlm`.
start_dir = Path.cwd().resolve()
project_root = next((p for p in [start_dir, *start_dir.parents] if (p / "vlm").is_dir()), start_dir)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from vlm.llava_client import query_llava_position

print(f"Project root: {project_root}")
print(f"Python executable: {sys.executable}")
print(f"CUDA available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "CUDA is required for this notebook (expected RTX 3070)."
device = "cuda"
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Project root: D:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project
Python executable: d:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project\final_project_env\Scripts\python.exe
CUDA available: True
Using device: cuda
GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [ ]:
# Cell 2: Reward-shaping logic and callbacks
PROMPT = "Observe the car. Is it on the LEFT slope, RIGHT slope, or BOTTOM? Reply with only one word."

def normalize_label(raw_text: str) -> str:
    # Keep letters only, then normalize to uppercase for robust matching.
    cleaned = re.sub(r"[^A-Za-z]", "", raw_text).upper()
    if "LEFT" in cleaned:
        return "LEFT"
    if "RIGHT" in cleaned:
        return "RIGHT"
    return "BOTTOM"

def label_to_phi(label: str) -> float:
    # Potential reward mapping from VLM label.
    if label == "LEFT":
        return 0.5
    if label == "RIGHT":
        return 1.0
    return 0.0

class VLMRewardShapingWrapper(gym.Wrapper):
    def __init__(self, env, sample_every_n: int = 128, gamma: float = 0.99):
        super().__init__(env)
        self.sample_every_n = sample_every_n
        self.gamma = gamma
        self.step_count = 0
        self.cached_label = "BOTTOM"
        self.cached_phi = 0.0

    def step(self, action):
        obs, reward_env, terminated, truncated, info = self.env.step(action)
        self.step_count += 1

        # Update VLM reward only every N steps, then reuse it in between.
        if self.step_count % self.sample_every_n == 0:
            frame = self.env.render()
            llava_reply = query_llava_position(frame=frame, model="llava:7b", prompt=PROMPT)
            self.cached_label = normalize_label(llava_reply)
            self.cached_phi = label_to_phi(self.cached_label)
            print(
                f"[VLM] step={self.step_count}, raw='{llava_reply}', label={self.cached_label}, phi={self.cached_phi:.2f}"
            )

        # Shaped reward with cached potential: R_total = R_env + gamma * Phi(s').
        reward_total = reward_env + self.gamma * self.cached_phi
        info["reward_env"] = reward_env
        info["phi"] = self.cached_phi
        info["vlm_label"] = self.cached_label
        info["reward_total"] = reward_total
        return obs, reward_total, terminated, truncated, info

class StopOnFirstSuccessCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)

    def _on_step(self) -> bool:
        # Stop as soon as a high-quality episode reward is better than -160.
        for info in self.locals.get("infos", []):
            if "episode" in info:
                reward = info["episode"]["r"]
                if reward > -160:
                    print(f"High-quality success reached. Episode reward: {reward}. Stopping training.")
                    return False
        return True

class ReachRewardThresholdCallback(BaseCallback):
    def __init__(self, target_reward: float = -160.0, verbose=0):
        super().__init__(verbose)
        self.target_reward = target_reward
        self.first_reach_timestep = None

    def _on_step(self) -> bool:
        # Record the first timestep when target reward is reached.
        if self.first_reach_timestep is not None:
            return True
        for info in self.locals.get("infos", []):
            if "episode" in info:
                episode_reward = info["episode"]["r"]
                if episode_reward >= self.target_reward:
                    self.first_reach_timestep = self.num_timesteps
                    print(
                        f"Reached target reward {self.target_reward} at timestep {self.first_reach_timestep}."
                    )
                    break
        return True

In [ ]:
# Cell 3: Train, compare with baseline, and export a demo video
log_dir = Path("logs_and_results")
log_dir.mkdir(parents=True, exist_ok=True)

sample_every_n = 128
gamma = 0.99
total_timesteps_cap = 100_000_000
baseline_timesteps_to_minus_160 = 240_000

base_env = gym.make("MountainCar-v0", render_mode="rgb_array")
monitored_env = Monitor(base_env)
train_env = VLMRewardShapingWrapper(monitored_env, sample_every_n=sample_every_n, gamma=gamma)

model = PPO(
    policy="MlpPolicy",
    env=train_env,
    verbose=1,
    tensorboard_log=str(log_dir / "tensorboard_vlm_first_success"),
    device="cuda",
)

threshold_callback = ReachRewardThresholdCallback(target_reward=-160.0)
stop_callback = StopOnFirstSuccessCallback()
callbacks = CallbackList([threshold_callback, stop_callback])

model.learn(
    total_timesteps=total_timesteps_cap,
    callback=callbacks,
    tb_log_name="ppo_mountain_car_vlm_first_success",
)

model_path = log_dir / "mountain_car_vlm_first_success"
model.save(str(model_path))
vlm_final_steps = model.num_timesteps
train_env.close()
print("Model saved to logs_and_results/mountain_car_vlm_first_success.zip")

baseline_steps_file = log_dir / "baseline_final_steps.txt"
if baseline_steps_file.exists():
    baseline_final_steps = int(baseline_steps_file.read_text(encoding="utf-8").strip())
else:
    baseline_final_steps = baseline_timesteps_to_minus_160
    print("Baseline step file not found. Using fallback baseline value: 240000")

print(f"Final Step Count (Baseline): {baseline_final_steps}")
print(f"Final Step Count (VLM-PPO): {vlm_final_steps}")
step_reduction_ratio = (baseline_final_steps - vlm_final_steps) / baseline_final_steps
print(f"Step Reduction Ratio: {step_reduction_ratio:.2%}")

if threshold_callback.first_reach_timestep is None:
    print("Did not reach reward -160 before stop.")
else:
    ours_steps_to_minus_160 = threshold_callback.first_reach_timestep
    speedup = baseline_final_steps / ours_steps_to_minus_160
    print(f"VLM-PPO steps to -160: {ours_steps_to_minus_160}")
    print(f"Speedup to -160 vs baseline final steps: {speedup:.2f}x")

video_env = gym.make("MountainCar-v0", render_mode="rgb_array")
loaded_model = PPO.load(str(model_path.with_suffix(".zip")), device="cuda")

frames = []
obs, info = video_env.reset()
terminated = False
truncated = False
while not (terminated or truncated):
    frames.append(video_env.render())
    action, _ = loaded_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = video_env.step(action)

video_env.close()
video_path = log_dir / "mountain_car_vlm_first_success_demo.mp4"
imageio.mimsave(video_path, frames, fps=30)
Video(str(video_path), embed=True)

Using cuda device
Wrapping the env in a DummyVecEnv.
Logging to logs_and_results\tensorboard_vlm_first_success\ppo_mountain_car_vlm_first_success_1


d:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project\final_project_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


[VLM] step=16, raw='Left', label=LEFT, phi=0.50
[VLM] step=32, raw='Left', label=LEFT, phi=0.50
[VLM] step=48, raw='Left', label=LEFT, phi=0.50
[VLM] step=64, raw='Left', label=LEFT, phi=0.50
[VLM] step=80, raw='Left', label=LEFT, phi=0.50
[VLM] step=96, raw='Right', label=RIGHT, phi=1.00
[VLM] step=112, raw='Left', label=LEFT, phi=0.50
[VLM] step=128, raw='Left', label=LEFT, phi=0.50
[VLM] step=144, raw='Left', label=LEFT, phi=0.50
[VLM] step=160, raw='Left', label=LEFT, phi=0.50
[VLM] step=176, raw='Right', label=RIGHT, phi=1.00
[VLM] step=192, raw='Left', label=LEFT, phi=0.50
[VLM] step=208, raw='Left', label=LEFT, phi=0.50
[VLM] step=224, raw='Right', label=RIGHT, phi=1.00
[VLM] step=240, raw='Left', label=LEFT, phi=0.50
[VLM] step=256, raw='Left', label=LEFT, phi=0.50
[VLM] step=272, raw='Right', label=RIGHT, phi=1.00
[VLM] step=288, raw='Right', label=RIGHT, phi=1.00
[VLM] step=304, raw='Left', label=LEFT, phi=0.50
[VLM] step=320, raw='Right', label=RIGHT, phi=1.00
[VLM] step=336

KeyboardInterrupt: 